Infer the value of applying a customer takeover at_risk evaluation tool.

This uses the method outlined in `sampler_doc` by John Mount, Nina Zumel Win Vector LLC (September 2026).

 https://www.iii.org/fact-statistic/facts-statistics-identity-theft-and-cybercrime

In [1]:
# imports and config
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az
from pathlib import Path
import scipy.stats
from tqdm import tqdm

rng = np.random.default_rng()

Estimated facts about the scanner.

In [2]:
calibration_possible_positives = 10
calibration_true_positives = 8
calibration_possible_negatives = 50
calibration_true_negatives = 48

In [3]:
# sensitivity
sensitivity = (calibration_true_positives + 1) / (calibration_possible_positives + 2)
# specificity
specificity = (calibration_true_negatives + 1) / (calibration_possible_negatives + 2)
(sensitivity, specificity)

(0.75, 0.9423076923076923)

Estimated facts about the environment.

In [4]:
n_compromised_last_year = 45
n_accounts_last_year = 19000

Assumed facts about the new code base to scan.

In [5]:
# known
n_accounts = 20000

In [6]:
# not known prior to scan (use to limit number of simulated counter-argument scans)
study_rate = 0.02
study_rate

0.02

In [7]:
modeled_at_risk_rates = tuple([study_rate + 0.005 * d for d in range(-2, 3)])
modeled_at_risk_rates

(0.01, 0.015, 0.02, 0.025, 0.03)

Cost and utility factors.

In [8]:
# cost and utility factors
scan_cost = 5.00
intervention_cost = 100.00
intervention_effectiveness = 0.80
expected_loss = 12000.00   # Typical of credit cards: https://www.pingidentity.com/en-us/docs/blogs/account-takeover-ato-fraud
sd_loss = expected_loss / 3.0  # arbitrary
last_year_loss = np.round(rng.normal(
    n_compromised_last_year * expected_loss,
    np.sqrt(n_compromised_last_year) * sd_loss,
), 2)
last_year_loss

np.float64(541989.51)

Pick save dir.

In [9]:

output_dir = Path(f"account_example_n{n_accounts}")
output_dir.mkdir(exist_ok=True)
output_dir 

PosixPath('account_example_n20000')

Report config.

In [10]:

config = pd.DataFrame({
        'calibration_possible_positives': [calibration_possible_positives],
        'calibration_true_positives': [calibration_true_positives],
        'calibration_possible_negatives': [calibration_possible_negatives],
        'calibration_true_negatives': [calibration_true_negatives],
        'last_year_loss': [last_year_loss],
        'n_compromised_last_year': [n_compromised_last_year],
        'n_accounts_last_year': [n_accounts_last_year],
        'n_accounts': [n_accounts],
        'study_rate': [study_rate],
        'scan_cost': [scan_cost],
        'intervention_cost': [intervention_cost],
        'intervention_effectiveness': [intervention_effectiveness],
        'expected_loss': [expected_loss],
        'sd_loss': [sd_loss],
        'output_dir': [str(output_dir)],
})

config.T.style.hide(axis="columns")

calibration_possible_positives,10
calibration_true_positives,8
calibration_possible_negatives,50
calibration_true_negatives,48
last_year_loss,541989.510000
n_compromised_last_year,45
n_accounts_last_year,19000
n_accounts,20000
study_rate,0.020000
scan_cost,5.000000
intervention_cost,100.000000


Define a PyMC function that computes the log-probability of each true positive and false positive combination. And also define a scipy function that computes the same probabilities given the other parameters.

In [11]:
# numpy/scipy version of prob calc (faster for scaler/point work)
def softmax(x):
    """exp(x) / sum(exp(x))"""
    e_x = np.exp(x - np.max(x))  # For numerical stability
    return e_x / e_x.sum()

# define log probability of positives and negative portion of indications
def combo_logp(*, n_at_risk, modeled_sensitivity, n_safe, modeled_specificity, count_range, complement_range):
    """
    Arguments: PyMC model components.
    Return: vector of true positive count log-probabilities indexed by possibible number of true positives 
    """
    # for k indications to be peril, then presumed_indications - k are safe,
    # so assign the joint probability of this as the chance of seeing k indications
    # that turn out to be peril
    logp_of_k_indications_peril = pm.logp(  # log probalbity of seeing k true positives alone
        pm.Binomial.dist(
            n=n_at_risk, 
            p=modeled_sensitivity),
        count_range
    )
    logp_of_ck_indications_safe = pm.logp(  # log probabilty of seeing presumed_indications - k false postives alone
        pm.Binomial.dist(
            n=n_safe, 
            p=1 - modeled_specificity),
        complement_range  # complementary counts
    )
    return logp_of_k_indications_peril + logp_of_ck_indications_safe  # combine to get joint probability of both observations
    
# get probability of number of indications
def combo_p(*, n_at_risk, modeled_sensitivity, n_safe, modeled_specificity, count_range, complement_range):
    """
    Arguments: exact scalar values. 
    Return: vector of true positive count probabilities indexed by possibible number of true positives 
    """
    n_at_risk = int(n_at_risk)
    # for k indications to be peril, then presumed_indications - k are safe,
    # so assign the joint probability of this as the chance of seeing k indications
    # that turn out to be peril
    b_m = scipy.stats.Binomial(
        n=n_at_risk,
        p=modeled_sensitivity)
    logp_of_k_indications_peril = np.array([b_m.logpmf(v) for v in count_range], dtype=float)
    b_b = scipy.stats.Binomial(
        n=n_safe, 
        p=1 - modeled_specificity)
    logp_of_ck_indications_safe = np.array([b_b.logpmf(v) for v in complement_range], dtype=float)  # complementary counts
    return softmax(logp_of_k_indications_peril + logp_of_ck_indications_safe)

def run_model(*, presumed_indications: int, presumed_at_risk_rate: float):
    # Define the list of possible true positive counts and false positive counts.
    count_range = np.arange(presumed_indications + 1, dtype=int)
    complement_range = presumed_indications - count_range
    # Define the PyMC model and distribution relations and draw a sample.
    with pm.Model():
        modeled_sensitivity = pm.Beta(
            'modeled_sensitivity',
            alpha=1 + calibration_true_positives,
            beta=1 + calibration_possible_positives - calibration_true_positives,
        )
        modeled_specificity = pm.Beta(
            'modeled_specificity',
            alpha=1 + calibration_true_negatives,
            beta=1 + calibration_possible_negatives - calibration_true_negatives,
        )
        n_at_risk = pm.Binomial(
            "n_at_risk",
            n=n_accounts,
            p=presumed_at_risk_rate,
        )
        n_safe = pm.Deterministic(
            'n_safe',
            n_accounts - n_at_risk,
        )
        # enforce: n_true_positive (~binomial(n_at_risk, modeled_sensitivity) + 
        #          n_false_positive (~binomial(n_safe, 1 - modeled_specificity) = presumed_indications
        logp_of_k_peril_and_ck_safe = combo_logp(
            n_at_risk=n_at_risk, 
            modeled_sensitivity=modeled_sensitivity, 
            n_safe=n_safe, 
            modeled_specificity=modeled_specificity,
            count_range=count_range,
            complement_range=complement_range,
        )
        # sum out n_true_positive strategy, get probability of any legal observation
        marginal_logp = pm.math.logsumexp(logp_of_k_peril_and_ck_safe)
        pm.Potential("marginalized_obs", marginal_logp)
        idata = pm.sample(
            tune=10000, 
            draws=10000, 
            chains=4, 
            cores=4,
            target_accept=0.95,
        )
        return idata

In [12]:
def unpack_results(idata, *, presumed_indications: int, presumed_at_risk_rate: float) -> pd.DataFrame:
    # unpack sample data
    continuous_vars = (
        'modeled_sensitivity', 'modeled_specificity',
        )
    integer_vars = ('n_at_risk', 'n_safe',)
    res = pd.DataFrame({
        nm: idata.posterior[nm].stack(sample=('chain', 'draw')).values
        for nm in continuous_vars + integer_vars
    })
    res['presumed_at_risk_rate'] = presumed_at_risk_rate
    res['presumed_indications'] = presumed_indications
    # sample down and shuffle rows (make Python non-vectorized steps faster)
    res = res.loc[rng.choice(res.shape[0], size=1000, replace=False), :].reset_index(drop=True, inplace=False)
    for nm in integer_vars:
        res[nm] = res[nm].astype(int)
        assert np.all(res[nm] >= 0)
    for nm in continuous_vars:
        assert np.all(res[nm] >= 0)
        assert np.all(res[nm] <= 1)
    assert np.all(res['n_safe'] + res['n_at_risk'] >= presumed_indications)
    # generate n_true_positive and unscaled log liklihood
    count_range = np.arange(presumed_indications + 1, dtype=int)
    complement_range = presumed_indications - count_range
    res['n_true_positive'] = 0
    print(" undoing sum-out")
    for i in tqdm(range(res.shape[0])):
        p_n_true_positive = combo_p(
                    n_at_risk = res.loc[i, 'n_at_risk'], 
                    modeled_sensitivity = res.loc[i, 'modeled_sensitivity'], 
                    n_safe = res.loc[i, 'n_safe'], 
                    modeled_specificity = res.loc[i, 'modeled_specificity'],
                    count_range=count_range,
                    complement_range=complement_range,
                )
        n_true_positive = int(rng.choice(
                count_range,
                p=p_n_true_positive,))
        res.loc[i, 'n_true_positive'] = n_true_positive
    # Unpack the rest of the confusion matrix and double check.
    # get the whole confusion matrix as columns
    res['n_false_positive'] = presumed_indications - res['n_true_positive']
    res['n_false_negative'] = res['n_at_risk'] - res['n_true_positive']
    res['n_true_negative'] = res['n_safe'] - res['n_false_positive']
    # type convert, check, and confirm
    for nm in (
        'n_safe', 'n_at_risk',
        'n_true_positive', 'n_false_positive', 'n_false_negative', 'n_true_negative'):
        res[nm] = res[nm].astype(int)
        assert np.all(res[nm] >= 0)
    assert np.all(res['n_safe'] + res['n_at_risk'] >= presumed_indications)
    assert np.all(res['n_true_positive'] + res['n_false_positive'] == presumed_indications)
    assert np.all(res['n_true_positive'] + res['n_false_negative'] == res['n_at_risk'])
    assert np.all(res['n_true_negative'] + res['n_false_positive'] == res['n_safe'])
    # adjoin in more variables
    print(" adjoining observations")
    for s_idx in tqdm(range(res.shape[0])):
        attacker_coverage_times_risk_rate = rng.beta(
                    a=1 + config.loc[0, 'n_compromised_last_year'],
                    b=1 + config.loc[0, 'n_accounts_last_year'] - config.loc[0, 'n_compromised_last_year'],
                )
        attacker_coverage = attacker_coverage_times_risk_rate / presumed_at_risk_rate
        res.loc[s_idx, 'attacker_coverage'] = attacker_coverage
        n_failed_treatment = rng.binomial(
            n=res.loc[s_idx, 'n_true_positive'],
            p=1 - intervention_effectiveness)
        res.loc[s_idx, 'n at risk | intervention'] = (
            res.loc[s_idx, 'n_false_negative'] + n_failed_treatment)
        n_attacked_either_scenario = rng.binomial(
            n=res.loc[s_idx, 'n_false_negative'] + n_failed_treatment,
            p=attacker_coverage)
        n_additional_attacked_no_intervention = rng.binomial(
            n=res.loc[s_idx, 'n_true_positive'] - n_failed_treatment,
            p=attacker_coverage
        )
        res.loc[s_idx, 'n at risk | no intervention'] = (
                        res.loc[s_idx, 'n_false_negative']
                        + res.loc[s_idx, 'n_true_positive'])
        res.loc[s_idx, 'n compromised | no intervention'] = (
            n_attacked_either_scenario + n_additional_attacked_no_intervention)
        res.loc[s_idx, 'n compromised | intervention'] = (
            n_attacked_either_scenario)
        res.loc[s_idx, 'no intervention losses'] = np.maximum(0, rng.normal(
            loc = expected_loss * res.loc[s_idx, 'n compromised | no intervention'],
            scale = sd_loss * np.sqrt(res.loc[s_idx, 'n compromised | no intervention']),
        ))
        res.loc[s_idx, 'post intervention losses'] = np.maximum(0, rng.normal(
            expected_loss * res.loc[s_idx, 'n compromised | intervention'],
            scale = sd_loss * np.sqrt(res.loc[s_idx, 'n compromised | intervention']),
        ))
    return res


In [13]:
results = []
for presumed_at_risk_rate in modeled_at_risk_rates:
    e_indications = (
        n_accounts * presumed_at_risk_rate * sensitivity
        + n_accounts * (1 - presumed_at_risk_rate) * (1 - specificity)
    )
    var_indications = (
                n_accounts * presumed_at_risk_rate * sensitivity * (1 - (presumed_at_risk_rate * sensitivity))
                + n_accounts * (1 - presumed_at_risk_rate) * (1 - specificity) * (1 - ((1 - presumed_at_risk_rate) * (1 - specificity)))
            )
    rounded_indications = int(np.round(e_indications / 100) * 100)
    for presumed_indications in (rounded_indications - 100, rounded_indications, rounded_indications + 100):
        display(f"starting presumed_indications: {presumed_indications}, presumed_at_risk_rate: {presumed_at_risk_rate:0.2%}")
        idata = run_model(presumed_indications=presumed_indications, presumed_at_risk_rate=presumed_at_risk_rate)
        res_i = unpack_results(idata, presumed_indications=presumed_indications, presumed_at_risk_rate=presumed_at_risk_rate)
        res_i['E[indications]'] = e_indications
        res_i['var[indications]'] = var_indications
        display(res_i.describe())
        results.append(res_i)
scan_perf = pd.concat(results, ignore_index=True)

'starting presumed_indications: 1200, presumed_at_risk_rate: 1.00%'

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>NUTS: [modeled_sensitivity, modeled_specificity]
>Metropolis: [n_at_risk]


Output()

Sampling 4 chains for 10_000 tune and 10_000 draw iterations (40_000 + 40_000 draws total) took 63 seconds.


 undoing sum-out


100%|██████████| 1000/1000 [00:42<00:00, 23.31it/s]


... (125 outputs left out)

100%|██████████| 1000/1000 [00:59<00:00, 16.84it/s]


 adjoining observations


100%|██████████| 1000/1000 [00:01<00:00, 796.09it/s]


,modeled_sensitivity,modeled_specificity,n_at_risk,n_safe,presumed_at_risk_rate,presumed_indications,n_true_positive,n_false_positive,n_false_negative,n_true_negative,attacker_coverage,n at risk | intervention,n at risk | no intervention,n compromised | no intervention,n compromised | intervention,no intervention losses,post intervention losses,E[indications],var[indications]
count,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03,1000.0,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03,1000.000000,1000.000000,1.000000e+03
mean,0.759307,0.935893,602.095000,19397.905000,3.000000e-02,1700.0,457.297000,1242.703000,144.798000,18155.202000,0.080645,236.184000,602.095000,48.661000,19.081000,5.841321e+05,228413.835155,1569.230769,1.496472e+03
std,0.117329,0.004197,24.751295,24.751295,1.388473e-17,0.0,74.359035,74.359035,71.129056,71.129056,0.011884,57.854726,24.751295,9.925048,6.892891,1.241982e+05,83793.128994,0.000000,4.549749e-13
min,0.333773,0.921224,533.000000,19322.000000,3.000000e-02,1700.0,188.000000,1062.000000,3.000000,17899.000000,0.047125,115.000000,533.000000,17.000000,4.000000,2.137256e+05,43967.875010,1569.230769,1.496472e+03
25%,0.684877,0.933156,584.750000,19380.000000,3.000000e-02,1700.0,407.000000,1190.000000,92.000000,18107.000000,0.072182,191.000000,584.750000,41.750000,14.000000,4.952698e+05,164699.094928,1569.230769,1.496472e+03
50%,0.772921,0.936241,601.000000,19399.000000,3.000000e-02,1700.0,463.000000,1237.000000,137.500000,18162.500000,0.080005,230.000000,601.000000,48.000000,18.000000,5.797601e+05,220859.572343,1569.230769,1.496472e+03
75%,0.847839,0.939014,620.000000,19415.250000,3.000000e-02,1700.0,510.000000,1293.000000,193.000000,18208.000000,0.089057,272.250000,620.000000,55.000000,23.000000,6.674018e+05,281021.370761,1569.230769,1.496472e+03
max,0.992097,0.948356,678.000000,19467.000000,3.000000e-02,1700.0,638.000000,1512.000000,401.000000,18297.000000,0.118078,451.000000,678.000000,81.000000,56.000000,1.008777e+06,655257.930102,1569.230769,1.496472e+03


Save results for later use.

In [14]:
# save all
dataframes = {
    "config": config,
    "scan_perf": scan_perf,
}

for name, df in dataframes.items():
    path = output_dir / f"{name}.csv"
    df.to_csv(
        path,
        index=False,
    )